# Laboratorio 07 - Regresion Logistica
## CC3074 - Mineria de Datos | Universidad del Valle de Guatemala

**Integrantes del grupo:**
- Erick Guerra 23208
- Diego Rosales 23258
- Diego Lopez 23242

**Dataset:** Airbnb Listings (`listings.RData`)

---
## Inicializacion y preprocesamiento (base MD-LAB06)

Se replica la carga y limpieza del laboratorio anterior para mantener consistencia en el pipeline y comparabilidad entre resultados.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadr

from sklearn.model_selection import train_test_split

print('Imports OK')

Imports OK


In [8]:
# Carga del dataset
rdata_path = Path('data/listings.RData')
result = pyreadr.read_r(str(rdata_path))
df = result['listings']
print(f'Dimensiones del dataset original: {df.shape}')

Dimensiones del dataset original: (171748, 80)


In [9]:
# Preprocesamiento (mismo enfoque de MD-LAB06)
df_eda = df.copy()

def parse_money(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(r'[$,]', '', regex=True).str.strip(),
        errors='coerce',
    )

if 'price' in df_eda.columns:
    df_eda['price_num'] = parse_money(df_eda['price'])

for col in ['host_response_rate', 'host_acceptance_rate']:
    if col in df_eda.columns:
        df_eda[f'{col}_num'] = pd.to_numeric(
            df_eda[col].astype(str).str.replace('%', '', regex=False),
            errors='coerce',
        )

candidate_features = [
    'room_type', 'property_type', 'neighbourhood_cleansed',
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'maximum_nights', 'availability_365',
    'number_of_reviews', 'reviews_per_month', 'review_scores_rating',
    'host_is_superhost', 'host_identity_verified',
    'host_response_rate_num', 'host_acceptance_rate_num',
    'calculated_host_listings_count', 'instant_bookable',
    'latitude', 'longitude', 'price_num',
]

selected_cols = [c for c in candidate_features if c in df_eda.columns]
model_df = df_eda[selected_cols].copy()
before_rows = model_df.shape[0]

model_df = model_df[model_df['price_num'].notna()].copy()
p99_price = model_df['price_num'].quantile(0.99)
model_df = model_df[model_df['price_num'] <= p99_price].copy()

num_cols = [c for c in model_df.select_dtypes(include=np.number).columns if c != 'price_num']
cat_cols = model_df.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

for col in num_cols:
    model_df[col] = model_df[col].fillna(model_df[col].median())

for col in cat_cols:
    model_df[col] = model_df[col].astype(str).replace('nan', 'SinDato')

X = model_df.drop(columns=['price_num'])
y_raw = model_df['price_num']

print(f'Filas iniciales: {before_rows:,}')
print(f'Filas finales:   {len(model_df):,}')
print(f'Num features:    {X.shape[1]}')

Filas iniciales: 171,748
Filas finales:   75,531
Num features:    21


---
## Actividad 2 - Uso de los mismos conjuntos de entrenamiento y prueba

Se conserva exactamente el mismo criterio de split usado en hojas anteriores: `test_size=0.2`, `random_state=42`, y estratificacion por deciles de precio (sobre `log1p(price_num)`).

In [10]:
# Split identico al laboratorio anterior
y_log = np.log1p(y_raw)
price_deciles = pd.qcut(y_log, q=10, labels=False, duplicates='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_raw,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=price_deciles,
)

print(f'Entrenamiento: {X_train.shape[0]:,} filas')
print(f'Prueba:        {X_test.shape[0]:,} filas')

Entrenamiento: 60,424 filas
Prueba:        15,107 filas


---
## Actividad 1 - Variables dicotomicas para la respuesta categorica

Se crea la variable respuesta categorica en tres niveles (`economica`, `media`, `cara`) y luego se generan tres variables dicotomicas (0/1):
- `es_economica`
- `es_media`
- `es_cara`

In [11]:
# Umbrales usando solo train (evita data leakage)
p33 = y_train.quantile(0.33)
p66 = y_train.quantile(0.66)

def make_price_cat(price_series: pd.Series, low: float, high: float) -> pd.Series:
    return pd.cut(
        price_series,
        bins=[-np.inf, low, high, np.inf],
        labels=['economica', 'media', 'cara'],
    )

y_cat_all = make_price_cat(y_raw, p33, p66)
y_cat_train = y_cat_all.loc[y_train.index]
y_cat_test = y_cat_all.loc[y_test.index]

y_dicot_train = pd.DataFrame({
    'es_cara': (y_cat_train == 'cara').astype(int),
    'es_media': (y_cat_train == 'media').astype(int),
    'es_economica': (y_cat_train == 'economica').astype(int),
}, index=y_cat_train.index)

y_dicot_test = pd.DataFrame({
    'es_cara': (y_cat_test == 'cara').astype(int),
    'es_media': (y_cat_test == 'media').astype(int),
    'es_economica': (y_cat_test == 'economica').astype(int),
}, index=y_cat_test.index)

print(f'Umbral economica/media (P33): {p33:.2f}')
print(f'Umbral media/cara     (P66): {p66:.2f}')
print('\nDistribucion de clases en train:')
print(y_cat_train.value_counts().sort_index())
print('\nPrimeras filas de variables dicotomicas (train):')
display(y_dicot_train.head())

Umbral economica/media (P33): 141.00
Umbral media/cara     (P66): 260.00

Distribucion de clases en train:
price_num
economica    20143
media        19792
cara         20489
Name: count, dtype: int64

Primeras filas de variables dicotomicas (train):


,es_cara,es_media,es_economica
23421,0,0,1
36797,0,0,1
25953,0,0,1
160615,0,1,0
27075,0,0,1


In [12]:
# Verificacion rapida: cada fila debe tener exactamente una clase activa
valid_train = (y_dicot_train.sum(axis=1) == 1).all()
valid_test = (y_dicot_test.sum(axis=1) == 1).all()

print(f'Validez dicotomicas train (una clase activa): {valid_train}')
print(f'Validez dicotomicas test  (una clase activa): {valid_test}')
print(f'Tamano X_train: {X_train.shape} | Tamano y_dicot_train: {y_dicot_train.shape}')
print(f'Tamano X_test : {X_test.shape} | Tamano y_dicot_test : {y_dicot_test.shape}')

Validez dicotomicas train (una clase activa): True
Validez dicotomicas test  (una clase activa): True
Tamano X_train: (60424, 21) | Tamano y_dicot_train: (60424, 3)
Tamano X_test : (15107, 21) | Tamano y_dicot_test : (15107, 3)
